In [22]:
from PyPDF2 import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chat_models import ChatOpenAI
from langchain.chains.question_answering import load_qa_chain
import warnings
import pandas as pd
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
import os
import openai
import re
from jobspy import scrape_jobs

In [2]:
pdf = 'content/resume.pdf'
pdf_reader = PdfReader(pdf)
print(pdf_reader)

In [3]:
text = ''
for page in pdf_reader.pages:
    text += page.extract_text()

print(text)

Andre Sealy
New York, NY |(347)-461-7821 |andretsealy@gmail.com |/ewww.kidquant.com |/gtbkidquant
EDUCATION
Stevens Institute of Technology New York, NY
Masters of Science in Financial Engineering; Major GPA: 3.81 Sept 2024 - Present
Relevant Coursework: Stochastic Calculus, Pricing & Hedging, Probability Theory, Machine Learning, Deep Learning
Hunter College New York, NY
Bachelor of Arts in Mathematics; Major GPA: 3.68 Jan 2020 - Present
Relevant Coursework: Numerical Analysis, Real Analysis, Mathematical Statistics, Linear Algebra
Pace University New York, NY
Bachelor of Business Administration in Finance; Major GPA: 4.0 December 2018
Bachelor of Arts in Economics; Major GPA: 4.0
Honors: Beta Gamma Sigma, Golden Key Society
EXPERIENCE
America On Tech New York, NY
Data Science Instructor Nov 2023 - Present
•Facilitate and coordinate weekly lectures, lab projects, coding examples, graded quizzes and homework assignments.
• Design interactive weekly learning modules for more than 50 stu

In [4]:
# split the long text into small chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=700,
                                               chunk_overlap=200,
                                               length_function=len)

chunks = text_splitter.split_text(text)
chunks

['Andre Sealy\nNew York, NY |(347)-461-7821 |andretsealy@gmail.com |/ewww.kidquant.com |/gtbkidquant\nEDUCATION\nStevens Institute of Technology New York, NY\nMasters of Science in Financial Engineering; Major GPA: 3.81 Sept 2024 - Present\nRelevant Coursework: Stochastic Calculus, Pricing & Hedging, Probability Theory, Machine Learning, Deep Learning\nHunter College New York, NY\nBachelor of Arts in Mathematics; Major GPA: 3.68 Jan 2020 - Present\nRelevant Coursework: Numerical Analysis, Real Analysis, Mathematical Statistics, Linear Algebra\nPace University New York, NY\nBachelor of Business Administration in Finance; Major GPA: 4.0 December 2018\nBachelor of Arts in Economics; Major GPA: 4.0',
 'Pace University New York, NY\nBachelor of Business Administration in Finance; Major GPA: 4.0 December 2018\nBachelor of Arts in Economics; Major GPA: 4.0\nHonors: Beta Gamma Sigma, Golden Key Society\nEXPERIENCE\nAmerica On Tech New York, NY\nData Science Instructor Nov 2023 - Present\n•Faci

In [5]:
chunks[0]

'Andre Sealy\nNew York, NY |(347)-461-7821 |andretsealy@gmail.com |/ewww.kidquant.com |/gtbkidquant\nEDUCATION\nStevens Institute of Technology New York, NY\nMasters of Science in Financial Engineering; Major GPA: 3.81 Sept 2024 - Present\nRelevant Coursework: Stochastic Calculus, Pricing & Hedging, Probability Theory, Machine Learning, Deep Learning\nHunter College New York, NY\nBachelor of Arts in Mathematics; Major GPA: 3.68 Jan 2020 - Present\nRelevant Coursework: Numerical Analysis, Real Analysis, Mathematical Statistics, Linear Algebra\nPace University New York, NY\nBachelor of Business Administration in Finance; Major GPA: 4.0 December 2018\nBachelor of Arts in Economics; Major GPA: 4.0'

In [6]:

load_dotenv()
openai.api_key = os.environ["OPENAI_API_KEY"]

def openai_function(openai_api_key, chunks, analyze):

    # Using OpenAI service for embedding
    embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)

    # Facebook AI Similarity Search library help us to convert text data to numerical vector
    vectorstores = FAISS.from_texts(chunks, embedding=embeddings)

    # compares the query and chunks, enabling the selection of the top 'K' most similiar chunks based on their similarity scores.
    docs = vectorstores.similarity_search(query=analyze, k=3)

    # creates an OpenAI object, using the ChatGPT 4 
    llm = ChatOpenAI(model='gpt-4o', api_key=openai_api_key)

    # question-answering (QA) pipeline, making use of the load_qa_chain function
    chain = load_qa_chain(llm=llm, chain_type='stuff')

    response = chain.run(input_documents=docs, question=analyze)
    return response



In [7]:
def resume_summary(query_with_chunks):
    query = f''' need to detailed summarization of below resume and finally conclude them

                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                {query_with_chunks}
                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

summary = resume_summary(query_with_chunks=chunks)
summary_result = openai_function(openai_api_key=openai.api_key, chunks=chunks, analyze=summary)
print(summary_result)

Andre Sealy is a highly educated individual currently based in New York, NY. He is pursuing a Masters of Science in Financial Engineering at Stevens Institute of Technology, maintaining a major GPA of 3.81, and is expected to complete it by September 2024. His coursework includes Stochastic Calculus, Pricing & Hedging, Probability Theory, Machine Learning, and Deep Learning. Andre also holds a Bachelor of Arts in Mathematics from Hunter College with a major GPA of 3.68, and a Bachelor of Business Administration in Finance as well as a Bachelor of Arts in Economics from Pace University, both with a perfect major GPA of 4.0. He received honors such as being a member of Beta Gamma Sigma and the Golden Key Society.

Professionally, Andre has experience as a Data Science Instructor at America On Tech, where he has been working since November 2023. In this role, he facilitates lectures and projects, focusing on Machine Learning, Statistics, and Data Visualization for over 50 students. Before

In [ ]:
def resume_strength(query_with_chunks):
    query = f'''need to detailed analysis and explain of the strength of below resume and finally conclude them
                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                {query_with_chunks}
                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

strengths = resume_strength(query_with_chunks=chunks)
strengths_result = openai_function(openai_api_key=openai.api_key, chunks=chunks, analyze=strengths)
print(strengths_result)

In [ ]:
def resume_weakness(query_with_chunks):
    query = f'''need to detailed analysis and explain of the weakness of below resume and how to improve make a better resume.

                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                {query_with_chunks}
                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

weakness = resume_weakness(query_with_chunks=summary_result)
result_weakness = openai_function(openai_api_key=openai.api_key, chunks=chunks, analyze=weakness)
print(result_weakness)

In [ ]:
def job_title_suggestion(query_with_chunks):

    query = f''' what are the job roles i apply to likedin based on below?
                  
                  """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                  {query_with_chunks}
                  """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

suggestion = job_title_suggestion(query_with_chunks=summary_result)
result_suggestion = openai_function(openai_api_key=openai.api_key, chunks=chunks, analyze=suggestion)
print(result_suggestion)

In [10]:
def job_title_prompt(query_with_chunks):
    query = f'''Based on my resume, come up with some job roles that would best fit my skills and abilities.
                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                {query_with_chunks}

                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

job_prompt = job_title_prompt(query_with_chunks=summary_result)
jobs = openai_function(openai_api_key=openai.api_key, chunks=chunks, analyze=job_prompt)
print(jobs)


Based on the information provided in your resume, here are some potential job roles that would best fit your skills and abilities:

1. **Quantitative Analyst**: Your strong background in financial engineering, mathematics, and machine learning makes you an ideal candidate for roles that involve developing and implementing quantitative models for trading, risk management, or investment strategies.

2. **Data Scientist in Finance**: With your experience in machine learning, data science, and financial markets, a role that focuses on analyzing financial data to develop predictive models and insights would be a great fit.

3. **Financial Engineer**: Your education and experience align well with roles that involve designing and implementing financial products or systems, particularly those using advanced quantitative and computational techniques.

4. **Machine Learning Engineer**: Given your expertise in machine learning frameworks such as TensorFlow and PyTorch, a role that involves buildi

In [12]:
jobs_list = re.findall(r"\*\*(.*?)\*\*", jobs)

Quantitative Analyst
Data Scientist in Finance
Financial Engineer
Machine Learning Engineer
Investment Research Analyst
Risk Management Analyst
Financial Data Scientist
Instructor or Educator in Data Science/Finance


In [20]:
jobs_scraped = scrape_jobs(
    site_name=[
        "indeed",
        "linkedin",
        "glassdoor",
        "google",
    ],
    search_term="Quantitative Analyst",
    location="New York, NY",
    max_results=20,
    country_indeed="USA",
)

jobs_scraped.head()

,id,site,job_url,job_url_direct,title,company,location,date_posted,job_type,salary_source,interval,min_amount,max_amount,currency,is_remote,job_level,job_function,listing_type,emails,description,company_industry,company_url,company_logo,company_url_direct,company_addresses,company_num_employees,company_revenue,company_description,skills,experience_range,company_rating,company_reviews_count,vacancy_count,work_from_home_type
0,gd-1009722344786,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,NaN,Senior Compensation Analyst,the NBA,United States,2025-04-25,NaN,direct_data,yearly,135000.0,150000.0,USD,False,NaN,None,organic,NaN,**WORK OPTION:** The NBA currently provides el...,NaN,https://www.glassdoor.com/Overview/W-EI_IE2908...,https://media.glassdoor.com/sql/2908/nba-squar...,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None
1,gd-1009722858798,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,NaN,Data Analyst(Non-IT),Capleo Global LLC,"New York, NY",2025-04-25,NaN,direct_data,hourly,30.0,30.0,USD,False,NaN,None,organic,NaN,**Pay:** $30/\-Hour\n \n \n\nShift; Mon\-Fri...,NaN,https://www.glassdoor.com/Overview/W-EI_IE7128...,https://media.glassdoor.com/sql/712840/global-...,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None
2,gd-1009722098356,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,NaN,Data Analyst- Physician Services Organization-...,NewYork-Presbyterian Hospital,Manhattan,2025-04-25,NaN,direct_data,yearly,70500.0,95000.0,USD,False,NaN,None,organic,NaN,"**Data Analyst– New York, NY \- Day**\n-------...",NaN,https://www.glassdoor.com/Overview/W-EI_IE1215...,https://media.glassdoor.com/sql/121522/newyork...,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None
3,gd-1009721946928,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,NaN,Quantitative Risk Analyst,Fidelity Investments,"Jersey City, NJ",2025-04-25,NaN,direct_data,yearly,58000.0,91000.0,USD,False,NaN,None,organic,NaN,### **Job Description:**\n\n**Quantitative Ris...,NaN,https://www.glassdoor.com/Overview/W-EI_IE2786...,https://media.glassdoor.com/sql/2786/fidelity-...,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None
4,gd-1009720720937,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,NaN,Operations Analyst,Comcast Corporation,"New York, NY",2025-04-24,NaN,direct_data,hourly,28.0,42.0,USD,False,NaN,None,organic,NaN,"Universal Ads, a part of Comcast, enables any ...",NaN,https://www.glassdoor.com/Overview/W-EI_IE1280...,https://media.glassdoor.com/sql/1280/comcast-s...,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None


In [39]:
# Loop over the strings in jobs_list and run the function scrape_jobs
for job in jobs_list:
    # Scrape jobs for each job title in jobs_list
    jobs_scraped_for_job = scrape_jobs(
        site_name=[
            "indeed",
            "linkedin",
            "glassdoor",
            "google",
        ],
        search_term=job,
        location="New York, NY",
        max_results=20,
        country_indeed="USA",
    )

    jobs_scraped_for_job['job_type'] = job

    # Merge the output with the previous dataframe
    jobs_scraped = pd.concat([jobs_scraped, jobs_scraped_for_job], ignore_index=True)


jobs_scraped = jobs_scraped[
    [
        # "id",
        "site",
        "job_url",
        "title",
        "company",
        "date_posted",
        "job_type",
        "interval",
        "min_amount",
        "max_amount",
        "currency",
        "description",
    ]
]

jobs_scraped.iloc[:25]

,site,job_url,title,company,date_posted,job_type,interval,min_amount,max_amount,currency,description
0,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,Senior Compensation Analyst,the NBA,2025-04-25,Data Scientist in Finance,yearly,135000.0,150000.0,USD,**WORK OPTION:** The NBA currently provides el...
1,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,Data Analyst(Non-IT),Capleo Global LLC,2025-04-25,Data Scientist in Finance,hourly,30.0,30.0,USD,**Pay:** $30/\-Hour\n \n \n\nShift; Mon\-Fri...
2,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,Data Analyst- Physician Services Organization-...,NewYork-Presbyterian Hospital,2025-04-25,Data Scientist in Finance,yearly,70500.0,95000.0,USD,"**Data Analyst– New York, NY \- Day**\n-------..."
3,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,Quantitative Risk Analyst,Fidelity Investments,2025-04-25,Data Scientist in Finance,yearly,58000.0,91000.0,USD,### **Job Description:**\n\n**Quantitative Ris...
4,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,Operations Analyst,Comcast Corporation,2025-04-24,Data Scientist in Finance,hourly,28.0,42.0,USD,"Universal Ads, a part of Comcast, enables any ..."
5,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,Global Research - Asset-Backed Securities - An...,JPMorganChase,2025-04-24,Data Scientist in Finance,yearly,100000.0,125000.0,USD,**JOB DESCRIPTION** \n\nJoin our team as an A...
6,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,Analyst-Compliance,Amex,2025-04-24,Data Scientist in Finance,yearly,55000.0,105000.0,USD,**You Lead the Way. We’ve Got Your Back.**\n\n...
7,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,Quantitative Analyst - Equity Derivatives,UBS,2025-04-23,Data Scientist in Finance,yearly,215000.0,245000.0,USD,United States \- New York\nQuantitative Analys...
8,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,Financial Analyst,NYU Langone Health,2025-04-23,Data Scientist in Finance,yearly,64350.0,75000.0,USD,**NYU Grossman School of Medicine** is one of ...
9,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,Business Analyst,Morgan Stanley,2025-04-21,Data Scientist in Finance,yearly,72000.0,111000.0,USD,**Job Description**\n-------------------\n\n##...
